In [ ]:
import os
import numpy as np
import pandas as pd
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

INTERMEDIATE_RESULTS_DIR = os.path.abspath('../output/intermediate_results')
print(f"Intermediate results will be saved to: {INTERMEDIATE_RESULTS_DIR}")
os.makedirs(INTERMEDIATE_RESULTS_DIR, exist_ok=True)

# Data Preparation

In [ ]:
input_file = os.path.abspath('../human_eval/annotator_results/annotator_results.csv')
print(f"Read from file: {input_file}")
df = pd.read_csv(input_file)

# sort out rounds for backup annotators
df = df[~df["participant.label"].isna()]
# set participant code as index
df = df.set_index("participant.code")
print(f"Number of participants: {len(df)}")

# get index of last round
max_page_index = df["participant._max_page_index"].iloc[0].item()
print(f"ID of last round: {max_page_index}")

# allocated sample files per participant
allocated_sample_files = df["participant.participant_sample_file"].to_dict()
print(f"Allocated sample files per participant: {allocated_sample_files}")

In [3]:
# make dict of templates to parse results for each round
col_templates = [
    'reference_game.{round_id}.player.target_image_index',
    'reference_game.{round_id}.player.selected_image_index',
    'reference_game.{round_id}.player.description_informativeness',
    'reference_game.{round_id}.player.is_correct',
    'reference_game.{round_id}.player.no_object_fits',
    'reference_game.{round_id}.player.multiple_objects_fit',
    'reference_game.{round_id}.player.item_id',
    'reference_game.{round_id}.player.target_id',
    'reference_game.{round_id}.player.description',
    'reference_game.{round_id}.player.system',
    'reference_game.{round_id}.player.condition',
    'reference_game.{round_id}.subsession.round_number'
]

col_template_dict = {
    ct.split('.')[-1]: ct for ct in col_templates
}

In [4]:
round_dfs = []

# iterate over rounds and extract relevant columns
for round_id in range(1, max_page_index + 1):
    round_df = pd.DataFrame()  # df for current round
    for key, ct in col_template_dict.items():
        col_name = ct.format(round_id=round_id)
        round_df[key] = df[col_name]
    round_dfs.append(round_df.reset_index())  # reset index to have participant code as column
    
# concatenate all round dataframes into one
evaluation_df = pd.concat(round_dfs).reset_index(drop=True)
# adjust types
evaluation_df = evaluation_df.astype({
    **{c: int for c in ["target_image_index", "selected_image_index", "description_informativeness"]},
    **{c: bool for c in ["is_correct", "no_object_fits", "multiple_objects_fit"]}
})
# keep the last part of the huggingface system name
evaluation_df.system = evaluation_df.system.map(lambda x: x.split("/")[-1])

# Inter-Annotator Agreement

In [5]:
# select items used to compute inter-annotator agreement (IAA)
annotation_count_per_item = evaluation_df.groupby("item_id").size()
iaa_items = annotation_count_per_item[annotation_count_per_item > 1].index.to_list()

# create dataframe for IAA items only
iaa_df = evaluation_df[evaluation_df["item_id"].isin(iaa_items)]
iaa_df = iaa_df.sort_values(by=["item_id", "participant.code"])
iaa_df = iaa_df.groupby("item_id").agg(list)

# sanity check: ensure that all participant codes are in the same order for each entry
assert all (iaa_df["participant.code"].map(str) == str(iaa_df["participant.code"].iloc[0]))

In [6]:
def kappa_interpretation(kappa):
    if kappa < 0:
        return "Poor agreement"
    elif kappa < 0.20:
        return "Slight agreement"
    elif kappa < 0.40:
        return "Fair agreement"
    elif kappa < 0.60:
        return "Moderate agreement"
    elif kappa < 0.80:
        return "Substantial agreement"
    else:
        return "Almost perfect agreement"

kappa_cols =  [
    "selected_image_index",
    "description_informativeness",
    "is_correct",
    "no_object_fits",
    "multiple_objects_fit",
]

kappa_df = pd.DataFrame(columns=["col", "categories", "kappa", "interpretation"]).set_index("col")

for col in kappa_cols:
    
    ratings = np.array(iaa_df[col].tolist())

    # aggregate_raters converts this "raw" format into counts per category
    # table shape: (n_subjects, n_categories)
    table, categories = aggregate_raters(ratings)

    # compute Fleiss' kappa
    kappa = fleiss_kappa(table, method='fleiss')
    
    # store results in kappa_df
    kappa_df.loc[col] = {"categories": categories, "kappa": kappa, "interpretation": kappa_interpretation(kappa)}
    
display(kappa_df)
    

,categories,kappa,interpretation
col,,,
selected_image_index,"[-1, 0, 1, 2]",0.774435,Substantial agreement
description_informativeness,"[-1, 1, 2, 3, 4, 5]",0.409006,Moderate agreement
is_correct,"[False, True]",0.651563,Substantial agreement
no_object_fits,"[False, True]",0.577965,Moderate agreement
multiple_objects_fit,"[False, True]",0.301363,Fair agreement


In [7]:
print(kappa_df.to_string())

                                      categories     kappa         interpretation
col                                                                              
selected_image_index               [-1, 0, 1, 2]  0.774435  Substantial agreement
description_informativeness  [-1, 1, 2, 3, 4, 5]  0.409006     Moderate agreement
is_correct                         [False, True]  0.651563  Substantial agreement
no_object_fits                     [False, True]  0.577965     Moderate agreement
multiple_objects_fit               [False, True]  0.301363         Fair agreement


In [8]:
print(kappa_df.to_latex())

\begin{tabular}{llrl}
\toprule
 & categories & kappa & interpretation \\
col &  &  &  \\
\midrule
selected_image_index & [-1  0  1  2] & 0.774435 & Substantial agreement \\
description_informativeness & [-1  1  2  3  4  5] & 0.409006 & Moderate agreement \\
is_correct & [False  True] & 0.651563 & Substantial agreement \\
no_object_fits & [False  True] & 0.577965 & Moderate agreement \\
multiple_objects_fit & [False  True] & 0.301363 & Fair agreement \\
\bottomrule
\end{tabular}



# Majority Vote for IAA Items

In [9]:
mode_columns = [
    "selected_image_index",
    "no_object_fits",
    "multiple_objects_fit",
]
avg_columns = ["description_informativeness"]
first_columns = [
    "target_image_index",
    "item_id",
    "target_id",
    "description",
    "system",
    "condition",
]

def mode(x):
    return x.mode()[0]

def round_mean_int(x):
    return int(np.mean(x).round().item())

In [10]:
# create a new dataframe for items that are not IAA items
no_iaa_df = evaluation_df[~evaluation_df["item_id"].isin(iaa_items)]

# iterate over each IAA item to aggregate the entries
iaa_item_dfs = []
for iaa_item in iaa_items: 
    # get all entries for the current IAA item
    iaa_item_entries = evaluation_df.loc[evaluation_df["item_id"] == iaa_item]

    # sanity check: ensure that all columns that should have a single value per item indeed have only one unique value
    for col in first_columns:
        unique_values = iaa_item_entries[col].unique()
        assert len(unique_values) == 1, f"Column '{col}' has multiple unique values: {unique_values}"

    # aggregate the entries for the IAA item
    iaa_item = iaa_item_entries.groupby("item_id", as_index=False).agg({
        **{c: mode for c in mode_columns}, # mode for categorical columns
        **{c: round_mean_int for c in avg_columns}, # mean for numerical columns
        **{c: "first" for c in first_columns} # first value for columns that have a single value per item
    })
    assert len(iaa_item) == 1, "There should be only one entry per item after aggregation."

    # compute is_correct based on target idx and mode of selected image indices
    iaa_item["is_correct"] = iaa_item.target_image_index == iaa_item.selected_image_index
    # set participant code to "aggregated" for IAA items
    iaa_item["participant.code"] = "aggregated"
    # placeholder for round number, not relevant for aggregated items
    iaa_item["round_number"] = -1
    
    # append the aggregated IAA item to the list
    iaa_item_dfs.append(iaa_item)

# combine the non-IAA items with the aggregated IAA items into a single dataframe
majority_vote_df = pd.concat([no_iaa_df, *iaa_item_dfs]).reset_index(drop=True)

# sanity checks for the combined dataframe
assert len(majority_vote_df.groupby(["system", "condition"]).size().unique()), "Each system-condition combination should have the same number of items."
assert all (majority_vote_df.groupby("item_id").size() == 1), "Each item_id should have exactly one entry in the majority_vote_df."

In [11]:
_majority_vote_df = majority_vote_df.copy()

# add columns for thinking and architecture based on the system name
_majority_vote_df["thinking"] = _majority_vote_df.system.map(lambda x: "_thinking" in x)
_majority_vote_df["architecture"] = _majority_vote_df.system.map(lambda x: x.split("_thinking")[0])

# set description_informativeness to NaN for items where it is less than 0 or where no_object_fits is True (cleanup for aggregation)
_majority_vote_df.loc[_majority_vote_df['description_informativeness'] < 0, 'description_informativeness'] = np.nan
_majority_vote_df.loc[_majority_vote_df.no_object_fits, 'description_informativeness'] = np.nan
# rescale description_informativeness to center around 0
# (negative: not enough information, 0: just enough information, positive: too much information)
_majority_vote_df["description_informativeness"] = _majority_vote_df.description_informativeness - 3

# set architecture as categorical with a specific order
arch_cat_type = pd.CategoricalDtype(
    categories=[
        "human",
        "Qwen3.6-27B-FP8",
        "Qwen3.5-4B",
        "Qwen3.5-9B",
        "Qwen3.5-27B-FP8",
        "gemma-4-E4B-it",
        "gemma-4-12B-it",
        "gemma-4-26B-A4B-it-FP8-Dynamic",
    ],
    ordered=True,
)
_majority_vote_df["architecture"] = _majority_vote_df.architecture.astype(arch_cat_type)

# set condition as categorical with a specific order
cond_cat_type = pd.CategoricalDtype(
    categories=[
        "FAR", "SPLIT", "CLOSE"
    ],
    ordered=True,
)
_majority_vote_df["condition"] = _majority_vote_df.condition.astype(cond_cat_type)


In [12]:
# save preprocessed data
_majority_vote_df.to_json(os.path.join(INTERMEDIATE_RESULTS_DIR, "human_eval_per_item.json"), orient="records")

In [13]:
_majority_vote_df

,participant.code,target_image_index,selected_image_index,description_informativeness,is_correct,no_object_fits,multiple_objects_fit,item_id,target_id,description,system,condition,round_number,thinking,architecture
0,nzh9w6g0,0,0,2.0,True,False,False,5704-eb65850a-01b8-43b7-ab26-937928c2f46c_57_Q...,5704-eb65850a-01b8-43b7-ab26-937928c2f46c_57,"Top row: grey, yellow, teal. Middle row: cyan,...",Qwen3.5-4B_thinking,FAR,1,True,Qwen3.5-4B
1,oqscvy0l,0,0,2.0,True,False,False,5207-e36d6a16-30b0-47b2-a969-743b0f0ffe1d_40_Q...,5207-e36d6a16-30b0-47b2-a969-743b0f0ffe1d_40,"The grid has a red square, a purple square, an...",Qwen3.5-9B_thinking,FAR,1,True,Qwen3.5-9B
2,kdwvwipq,0,0,2.0,True,False,False,9193-e80b9de9-7035-4132-a05d-786e925f9e1c_39_Q...,9193-e80b9de9-7035-4132-a05d-786e925f9e1c_39,The grid with a red square in the bottom left ...,Qwen3.5-27B-FP8,FAR,1,False,Qwen3.5-27B-FP8
3,nzh9w6g0,2,2,0.0,True,False,False,2362-28cbc35e-2542-495b-8053-8ab762f7dad9_54_g...,2362-28cbc35e-2542-495b-8053-8ab762f7dad9_54,"The top row is cyan, orange, and brown.",gemma-4-E4B-it_thinking,FAR,2,True,gemma-4-E4B-it
4,oqscvy0l,2,2,2.0,True,False,False,2589-050e0f91-7dba-48c6-b219-c1d2a9b4eb39_7_go...,2589-050e0f91-7dba-48c6-b219-c1d2a9b4eb39_7,"Top row is dark gray, orange, and blue; middle...",gemma-4-12B-it,FAR,2,False,gemma-4-12B-it
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2245,aggregated,2,2,2.0,True,False,False,9597-b2286185-8375-4c3a-81a1-842be70a921e_27_R...,9597-b2286185-8375-4c3a-81a1-842be70a921e_27,"A grid with purple, cyan, grey, dark blue, pin...",gemma-4-26B-A4B-it-FP8-Dynamic,FAR,-1,False,gemma-4-26B-A4B-it-FP8-Dynamic
2246,aggregated,2,2,2.0,True,False,False,9955-1cbd5506-e782-422f-84e7-70366d4f805c_15_Q...,9955-1cbd5506-e782-422f-84e7-70366d4f805c_15,"A grid with a white background, featuring a to...",Qwen3.5-4B_thinking,SPLIT,-1,True,Qwen3.5-4B
2247,aggregated,0,0,-1.0,True,False,False,9955-1cbd5506-e782-422f-84e7-70366d4f805c_33_g...,9955-1cbd5506-e782-422f-84e7-70366d4f805c_33,"Yellow, magenta, olive green, and slate purple.",gemma-4-E4B-it_thinking,SPLIT,-1,True,gemma-4-E4B-it
2248,aggregated,2,2,0.0,True,False,False,9955-1cbd5506-e782-422f-84e7-70366d4f805c_6_Qw...,9955-1cbd5506-e782-422f-84e7-70366d4f805c_6,The grid with a cyan square in the top-left co...,Qwen3.5-9B_thinking,CLOSE,-1,True,Qwen3.5-9B


# Analyze Data

In [14]:
aggregated_results = _majority_vote_df.groupby(["architecture", "thinking"]).agg({
    c: "mean" for c in ["description_informativeness", "is_correct", "no_object_fits", "multiple_objects_fit"]
}).sort_index(ascending=[True, False])

perc_columns = ["is_correct", "no_object_fits", "multiple_objects_fit"]
for column in perc_columns:
    aggregated_results[column] = (aggregated_results[column] * 100).round(2)
aggregated_results["description_informativeness"] = aggregated_results.description_informativeness.round(2)

aggregated_results = aggregated_results.rename(columns={
    **{c: f"{c} (%)" for c in perc_columns}
})

aggregated_results

description_informativeness  \
architecture                   thinking                                
human                          False                           -0.16   
Qwen3.6-27B-FP8                True                            -0.01   
                               False                           -0.12   
Qwen3.5-4B                     True                             1.02   
                               False                            1.08   
Qwen3.5-9B                     True                             0.71   
                               False                           -0.20   
Qwen3.5-27B-FP8                True                             0.02   
                               False                           -0.15   
gemma-4-E4B-it                 True                            -0.13   
                               False                           -0.61   
gemma-4-12B-it                 True                             1.25   
                               False                            1.35   
gemma-4-26B-A4B-it-FP8-Dynamic True                             1.26   
                               False                            1.03   

                                         is_correct (%)  no_object_fits (%)  \
architecture                   thinking                                       
human                          False              88.00                2.67   
Qwen3.6-27B-FP8                True               92.67                1.33   
                               False              81.33                4.00   
Qwen3.5-4B                     True               78.67               12.67   
                               False              47.33               48.00   
Qwen3.5-9B                     True               88.67                2.00   
                               False              56.67                8.00   
Qwen3.5-27B-FP8                True               93.33                0.67   
                               False              81.33                4.00   
gemma-4-E4B-it                 True               81.33                4.67   
                               False              46.00               10.00   
gemma-4-12B-it                 True               91.33                2.67   
                               False              94.00                0.00   
gemma-4-26B-A4B-it-FP8-Dynamic True               90.67                4.67   
                               False              90.67                1.33   

                                         multiple_objects_fit (%)  
architecture                   thinking                            
human                          False                         5.33  
Qwen3.6-27B-FP8                True                          3.33  
                               False                         8.67  
Qwen3.5-4B                     True                          4.00  
                               False                         1.33  
Qwen3.5-9B                     True                          6.67  
                               False                        16.67  
Qwen3.5-27B-FP8                True                          4.67  
                               False                         9.33  
gemma-4-E4B-it                 True                         10.00  
                               False                        30.00  
gemma-4-12B-it                 True                          3.33  
                               False                         3.33  
gemma-4-26B-A4B-it-FP8-Dynamic True                          3.33  
                               False                         4.67

In [15]:
print(aggregated_results.reset_index().to_latex(index=False, float_format="%.2f", column_format="llrrrr"))

\begin{tabular}{llrrrr}
\toprule
architecture & thinking & description_informativeness & is_correct (%) & no_object_fits (%) & multiple_objects_fit (%) \\
\midrule
human & False & -0.16 & 88.00 & 2.67 & 5.33 \\
Qwen3.6-27B-FP8 & True & -0.01 & 92.67 & 1.33 & 3.33 \\
Qwen3.6-27B-FP8 & False & -0.12 & 81.33 & 4.00 & 8.67 \\
Qwen3.5-4B & True & 1.02 & 78.67 & 12.67 & 4.00 \\
Qwen3.5-4B & False & 1.08 & 47.33 & 48.00 & 1.33 \\
Qwen3.5-9B & True & 0.71 & 88.67 & 2.00 & 6.67 \\
Qwen3.5-9B & False & -0.20 & 56.67 & 8.00 & 16.67 \\
Qwen3.5-27B-FP8 & True & 0.02 & 93.33 & 0.67 & 4.67 \\
Qwen3.5-27B-FP8 & False & -0.15 & 81.33 & 4.00 & 9.33 \\
gemma-4-E4B-it & True & -0.13 & 81.33 & 4.67 & 10.00 \\
gemma-4-E4B-it & False & -0.61 & 46.00 & 10.00 & 30.00 \\
gemma-4-12B-it & True & 1.25 & 91.33 & 2.67 & 3.33 \\
gemma-4-12B-it & False & 1.35 & 94.00 & 0.00 & 3.33 \\
gemma-4-26B-A4B-it-FP8-Dynamic & True & 1.26 & 90.67 & 4.67 & 3.33 \\
gemma-4-26B-A4B-it-FP8-Dynamic & False & 1.03 & 90.67 & 1.33 & 4.

### Far/Split/Close

In [16]:
aggregated_results = _majority_vote_df.groupby(["architecture", "thinking", "condition"]).agg({  # also groupby condition
    c: "mean" for c in ["description_informativeness", "is_correct", "no_object_fits", "multiple_objects_fit"]
}).sort_index(ascending=[True, False, True])

perc_columns = ["is_correct", "no_object_fits", "multiple_objects_fit"]
for column in perc_columns:
    aggregated_results[column] = (aggregated_results[column] * 100).round(1)
aggregated_results["description_informativeness"] = aggregated_results.description_informativeness.round(2)

aggregated_results = aggregated_results.rename(columns={
    **{c: f"{c} (%)" for c in perc_columns}
})

aggregated_results

description_informativeness  \
architecture                   thinking condition                                
human                          False    FAR                               0.00   
                                        SPLIT                            -0.19   
                                        CLOSE                            -0.28   
Qwen3.6-27B-FP8                True     FAR                               0.33   
                                        SPLIT                            -0.08   
                                        CLOSE                            -0.27   
                               False    FAR                               0.17   
                                        SPLIT                            -0.15   
                                        CLOSE                            -0.37   
Qwen3.5-4B                     True     FAR                               1.27   
                                        SPLIT                             1.09   
                                        CLOSE                             0.67   
                               False    FAR                               1.77   
                                        SPLIT                             1.00   
                                        CLOSE                             0.44   
Qwen3.5-9B                     True     FAR                               1.14   
                                        SPLIT                             0.86   
                                        CLOSE                             0.11   
                               False    FAR                              -0.15   
                                        SPLIT                             0.09   
                                        CLOSE                            -0.52   
Qwen3.5-27B-FP8                True     FAR                               0.24   
                                        SPLIT                             0.02   
                                        CLOSE                            -0.20   
                               False    FAR                               0.10   
                                        SPLIT                            -0.17   
                                        CLOSE                            -0.40   
gemma-4-E4B-it                 True     FAR                               0.08   
                                        SPLIT                            -0.13   
                                        CLOSE                            -0.33   
                               False    FAR                              -0.23   
                                        SPLIT                            -0.54   
                                        CLOSE                            -1.04   
gemma-4-12B-it                 True     FAR                               1.62   
                                        SPLIT                             1.27   
                                        CLOSE                             0.85   
                               False    FAR                               1.82   
                                        SPLIT                             1.34   
                                        CLOSE                             0.90   
gemma-4-26B-A4B-it-FP8-Dynamic True     FAR                               1.62   
                                        SPLIT                             1.27   
                                        CLOSE                             0.87   
                               False    FAR                               1.51   
                                        SPLIT                             1.12   
                                        CLOSE                             0.48   

                                                   is_correct (%)  \
architecture                   thinking condition                   
human                          False    FAR               